# Ensemble Models — ELEC3612 Assignment 1

This notebook implements ensemble methods and compares their performance 
against the individual base models trained in the supervised models notebook.
We implement Bagging, Boosting, a Voting Classifier, and Stacking.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, BaggingClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier,
                              VotingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Load dataset
df = pd.read_csv('../data/aug_train.csv')
df = df.drop(columns=['enrollee_id'])

X = df.drop(columns=['target'])
y = df['target']

# Column types
numerical_cols = ['city_development_index', 'training_hours']
ordinal_cols = ['experience', 'last_new_job']
nominal_cols = ['city', 'gender', 'relevent_experience', 'enrolled_university',
                'education_level', 'major_discipline', 'company_size', 'company_type']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:\n{y.value_counts(normalize=True).round(3)}")

X shape: (19158, 12)
y shape: (19158,)

Class distribution:
target
0.0    0.751
1.0    0.249
Name: proportion, dtype: float64


## Setup

We reuse the same preprocessing pipeline and build_pipeline helper from 
the supervised models notebook. All ensemble models are evaluated on the 
same 80/20 stratified split for a fair comparison.

In [2]:
def build_pipeline(model):
    numerical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    ordinal_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    nominal_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    preprocessor = ColumnTransformer([
        ('num', numerical_pipeline, numerical_cols),
        ('ord', ordinal_pipeline, ordinal_cols),
        ('nom', nominal_pipeline, nominal_cols)
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

def evaluate(name, pipe, X_train, X_test, y_train, y_test):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    return {
        'model': name,
        'accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'precision': round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'recall':    round(recall_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'f1':        round(f1_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'y_pred':    y_pred
    }

# Standard 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (15326, 12), X_test: (3832, 12)


## Bagging

In [3]:
# Bagging with Decision Tree as base estimator
bagging = build_pipeline(
    BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
)

results = []
res = evaluate('Bagging (DT)', bagging, X_train, X_test, y_train, y_test)
results.append(res)

print(f"Accuracy : {res['accuracy']}")
print(f"Precision: {res['precision']}")
print(f"Recall   : {res['recall']}")
print(f"F1       : {res['f1']}")

Accuracy : 0.7675
Precision: 0.7538
Recall   : 0.7675
F1       : 0.7586


## Boosting

In [4]:
# AdaBoost
adaboost = build_pipeline(
    AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=100,
        random_state=42
    )
)
res_ada = evaluate('AdaBoost', adaboost, X_train, X_test, y_train, y_test)
results.append(res_ada)

# Gradient Boosting
gb = build_pipeline(
    GradientBoostingClassifier(n_estimators=100, random_state=42)
)
res_gb = evaluate('Gradient Boosting', gb, X_train, X_test, y_train, y_test)
results.append(res_gb)

print("AdaBoost:")
print(f"  Accuracy : {res_ada['accuracy']}")
print(f"  Precision: {res_ada['precision']}")
print(f"  Recall   : {res_ada['recall']}")
print(f"  F1       : {res_ada['f1']}")

print("\nGradient Boosting:")
print(f"  Accuracy : {res_gb['accuracy']}")
print(f"  Precision: {res_gb['precision']}")
print(f"  Recall   : {res_gb['recall']}")
print(f"  F1       : {res_gb['f1']}")

AdaBoost:
  Accuracy : 0.7769
  Precision: 0.7528
  Recall   : 0.7769
  F1       : 0.7503

Gradient Boosting:
  Accuracy : 0.7858
  Precision: 0.7719
  Recall   : 0.7858
  F1       : 0.7757


## Voting Classifier

In [5]:
# Voting Classifier (hard and soft)
voting_hard = build_pipeline(
    VotingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('lr', LogisticRegression(max_iter=1000, random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
        ],
        voting='hard'
    )
)
res_vh = evaluate('Voting (hard)', voting_hard, X_train, X_test, y_train, y_test)
results.append(res_vh)

voting_soft = build_pipeline(
    VotingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('lr', LogisticRegression(max_iter=1000, random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
        ],
        voting='soft'
    )
)
res_vs = evaluate('Voting (soft)', voting_soft, X_train, X_test, y_train, y_test)
results.append(res_vs)

print("Voting Hard:")
print(f"  Accuracy : {res_vh['accuracy']}")
print(f"  Precision: {res_vh['precision']}")
print(f"  Recall   : {res_vh['recall']}")
print(f"  F1       : {res_vh['f1']}")

print("\nVoting Soft:")
print(f"  Accuracy : {res_vs['accuracy']}")
print(f"  Precision: {res_vs['precision']}")
print(f"  Recall   : {res_vs['recall']}")
print(f"  F1       : {res_vs['f1']}")

Voting Hard:
  Accuracy : 0.7704
  Precision: 0.7521
  Recall   : 0.7704
  F1       : 0.7569

Voting Soft:
  Accuracy : 0.7518
  Precision: 0.7386
  Recall   : 0.7518
  F1       : 0.7438


## Stacking

In [6]:
# Stacking Classifier
stacking = build_pipeline(
    StackingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)),
            ('lr', LogisticRegression(max_iter=1000, random_state=42)),
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
        ],
        final_estimator=LogisticRegression(max_iter=1000, random_state=42),
        cv=5,
        n_jobs=-1
    )
)
res_stack = evaluate('Stacking', stacking, X_train, X_test, y_train, y_test)
results.append(res_stack)

print("Stacking:")
print(f"  Accuracy : {res_stack['accuracy']}")
print(f"  Precision: {res_stack['precision']}")
print(f"  Recall   : {res_stack['recall']}")
print(f"  F1       : {res_stack['f1']}")

Stacking:
  Accuracy : 0.7813
  Precision: 0.7608
  Recall   : 0.7813
  F1       : 0.7619


## Comparison with Base Models

We compare all ensemble methods against the individual base models from 
the supervised models notebook.

In [7]:
# Summary: ensemble vs base models
base_models = [
    {'model': 'Decision Tree (tuned)',     'accuracy': 0.7863, 'precision': 0.7721, 'recall': 0.7863, 'f1': 0.7758},
    {'model': 'Decision Tree (no tuning)','accuracy': 0.7035, 'precision': 0.7079, 'recall': 0.7035, 'f1': 0.7056},
    {'model': 'Logistic Regression',       'accuracy': 0.7777, 'precision': 0.7548, 'recall': 0.7777, 'f1': 0.7547},
    {'model': 'Random Forest (200 trees)', 'accuracy': 0.7701, 'precision': 0.7541, 'recall': 0.7701, 'f1': 0.7590},
    {'model': 'SVM (RBF)',                 'accuracy': 0.7784, 'precision': 0.7567, 'recall': 0.7784, 'f1': 0.7577},
]

ensemble_rows = [{'model': r['model'], 'accuracy': r['accuracy'],
                  'precision': r['precision'], 'recall': r['recall'],
                  'f1': r['f1']} for r in results]

summary_df = pd.DataFrame(base_models + ensemble_rows)
summary_df['type'] = ['Base'] * len(base_models) + ['Ensemble'] * len(ensemble_rows)
summary_df = summary_df.sort_values('f1', ascending=False).reset_index(drop=True)

print("=== Ensemble vs Base Models — Summary ===")
print(summary_df[['type', 'model', 'accuracy', 'precision', 'recall', 'f1']].to_string(index=False))

=== Ensemble vs Base Models — Summary ===
    type                     model  accuracy  precision  recall     f1
    Base     Decision Tree (tuned)    0.7863     0.7721  0.7863 0.7758
Ensemble         Gradient Boosting    0.7858     0.7719  0.7858 0.7757
Ensemble                  Stacking    0.7813     0.7608  0.7813 0.7619
    Base Random Forest (200 trees)    0.7701     0.7541  0.7701 0.7590
Ensemble              Bagging (DT)    0.7675     0.7538  0.7675 0.7586
    Base                 SVM (RBF)    0.7784     0.7567  0.7784 0.7577
Ensemble             Voting (hard)    0.7704     0.7521  0.7704 0.7569
    Base       Logistic Regression    0.7777     0.7548  0.7777 0.7547
Ensemble                  AdaBoost    0.7769     0.7528  0.7769 0.7503
Ensemble             Voting (soft)    0.7518     0.7386  0.7518 0.7438
    Base Decision Tree (no tuning)    0.7035     0.7079  0.7035 0.7056
